In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import wilcoxon

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/RF/ANDROIDS"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["file_stem"] = df["file_stem"].astype(str).str.strip()
df["depressed"] = pd.to_numeric(df["depressed"], errors="coerce")

feature_cols = [f"mfcc_{i}" for i in range(1, 14)] + ["pitch_mean", "energy_mean"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["depressed", "file_stem"]
df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols].astype(float).values
y = df_clean["depressed"].astype(int).values
groups = df_clean["file_stem"].values

print("Rows:", len(df_clean))
print("Groups:", df_clean["file_stem"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV + simple GridSearchCV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid = {
    "rf__n_estimators": [200, 500],
    "rf__max_depth": [None, 10],
    "rf__max_features": ["sqrt"],
    "rf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=gkf,
    n_jobs=-1
)
grid.fit(X, y, groups=groups)

best_rf_params = {
    k.replace("rf__", ""): v
    for k, v in grid.best_params_.items()
    if k.startswith("rf__")
}
print("Best RF params from GridSearchCV:", best_rf_params)

fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        **best_rf_params,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Paired Wilcoxon: |y - p_RF| vs |y - p_stratified_dummy| (same test samples)
    dummy = DummyClassifier(strategy="stratified", random_state=42)
    dummy.fit(X_train, y_train)
    p_dummy = dummy.predict_proba(X_test)[:, 1]
    e_rf = np.abs(y_test - y_proba)
    e_du = np.abs(y_test - p_dummy)
    if len(e_rf) >= 2 and not np.allclose(e_rf, e_du):
        w_res = wilcoxon(e_rf, e_du, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    Distributions of calibration errors differ (alpha=0.05).")
    else:
        print("    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "wilcoxon_p_vs_stratified_dummy": wilcoxon_p
    })

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["file_stem"].nunique(),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "androids_rf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "androids_rf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 224
Groups: 115
Features used: ['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'pitch_mean', 'energy_mean']
Best RF params from GridSearchCV: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 200}

Processing fold 1
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 1: 0.5162694825127413
    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).

Processing fold 2
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 2: 0.09929969887750739
    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).

Processing fold 3
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 3: 0.35165766560492173
    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).

Processing fold 4
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 4: 0.19615668222665816
    No significant difference vs stratified-dummy at alpha=0.05 

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import wilcoxon

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/RF/RADAR"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["participant_id"] = df["participant_id"].astype(str).str.strip()
df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")

# Binary target
df["depressed"] = (df["phq8_score"] >= 10).astype(int)

# RADAR feature columns
meta_cols = [
    "File", "participant_id", "Dataset", "Language", "Task",
    "recording_date", "Age", "Gender", "Education_Years",
    "Height", "phq8_score", "source_file", "depressed"
]

feature_cols = [c for c in df.columns if c not in meta_cols]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["depressed", "participant_id"]
df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols].astype(float).values
y = df_clean["depressed"].astype(int).values
groups = df_clean["participant_id"].values

print("Rows:", len(df_clean))
print("Groups:", df_clean["participant_id"].nunique())
print("Features used:", feature_cols)

# -----------------------------
# GroupKFold CV + simple GridSearchCV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid = {
    "rf__n_estimators": [200, 500],
    "rf__max_depth": [None, 10],
    "rf__max_features": ["sqrt"],
    "rf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=gkf,
    n_jobs=-1
)
grid.fit(X, y, groups=groups)

best_rf_params = {
    k.replace("rf__", ""): v
    for k, v in grid.best_params_.items()
    if k.startswith("rf__")
}
print("Best RF params from GridSearchCV:", best_rf_params)

fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = RandomForestClassifier(
        **best_rf_params,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Paired Wilcoxon: |y - p_RF| vs |y - p_stratified_dummy| (same test samples)
    dummy = DummyClassifier(strategy="stratified", random_state=42)
    dummy.fit(X_train, y_train)
    p_dummy = dummy.predict_proba(X_test)[:, 1]
    e_rf = np.abs(y_test - y_proba)
    e_du = np.abs(y_test - p_dummy)
    if len(e_rf) >= 2 and not np.allclose(e_rf, e_du):
        w_res = wilcoxon(e_rf, e_du, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    Distributions of calibration errors differ (alpha=0.05).")
    else:
        print("    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "wilcoxon_p_vs_stratified_dummy": wilcoxon_p
    })

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["participant_id"].nunique(),
    "accuracy_mean": fold_results_df["accuracy"].mean(),
    "accuracy_std": fold_results_df["accuracy"].std(),
    "f1_mean": fold_results_df["f1"].mean(),
    "f1_std": fold_results_df["f1"].std(),
    "roc_auc_mean": fold_results_df["roc_auc"].mean(),
    "roc_auc_std": fold_results_df["roc_auc"].std()
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "radar_rf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "radar_rf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 8515
Groups: 274
Features used: ['Clip_Duration', 'Speaking_Rate', 'Articulation_Rate', 'Phonation_Ratio', 'Pause_Rate', 'Pause_Ratio', 'mean_F0', 'stdev_F0_Semitone', 'HNR_dB', 'Spectral_Slope', 'Spectral_Tilt', 'Cepstral_Peak_Prominence', 'mean_F1_Loc', 'std_F1_Loc', 'mean_B1_Loc', 'std_B1_Loc', 'mean_F2_Loc', 'std_F2_Loc', 'mean_B2_Loc', 'std_B2_Loc', 'Spectral_Gravity', 'Spectral_Std_Dev']
Best RF params from GridSearchCV: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 500}

Processing fold 1
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 1: 0.01757085058005822
    Distributions of calibration errors differ (alpha=0.05).

Processing fold 2
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 2: 0.9800712392955155
    No significant difference vs stratified-dummy at alpha=0.05 (could be chance).

Processing fold 3
  Wilcoxon p (|y - p_RF| vs |y - p_dummy|), fold 3: 0.0422916207806747
    Distributions of calibration errors differ (alpha=0.05).


MERF for RADAR data


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

!pip install merf
!pip install sklearn.metrics
from sklearn.metrics import mean_squared_error
from scipy.stats import wilcoxon

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/MERF/RADAR"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean data
# -----------------------------
df["recording_date"] = pd.to_datetime(df["recording_date"], errors="coerce")
df["participant_id"] = df["participant_id"].astype(str).str.strip()

# -----------------------------
# Outcome + grouping
# -----------------------------
y_score = df["phq8_score"]
ID_clusters = df["participant_id"]

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = [
    "Age",
    "Gender",
    "Education_Years",
    "Height"
]

z_features = [c for c in z_candidates if c in df.columns]
Z_factors = df[z_features].copy()

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
x_candidates = [
    "Speaking_Rate",
    "Articulation_Rate",
    "Phonation_Ratio",
    "Pause_Rate",
    "Pause_Ratio",
    "mean_F0",
    "stdev_F0_Semitone",
    "HNR_dB",
    "Spectral_Slope",
    "Spectral_Tilt",
    "Cepstral_Peak_Prominence",
    "mean_F1_Loc",
    "std_F1_Loc",
    "mean_B1_Loc",
    "std_B1_Loc",
    "mean_F2_Loc",
    "std_F2_Loc",
    "mean_B2_Loc",
    "std_B2_Loc",
    "Spectral_Gravity",
    "Spectral_Std_Dev"
]

feature_cols = [c for c in x_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + z_features + ["phq8_score", "participant_id"]

df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols]
Z_factors = df_clean[z_features]
y_score = df_clean["phq8_score"]
ID_clusters = df_clean["participant_id"]

print("Rows:", len(df_clean))
print("Participants:", df_clean["participant_id"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV + simple GridSearchCV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "rf__n_estimators": [200, 500],
    "rf__max_depth": [None, 10],
    "rf__max_features": ["sqrt"],
    "rf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    n_jobs=-1
)
grid.fit(X, y_score, groups=ID_clusters)

best_rf_params = {
    k.replace("rf__", ""): v
    for k, v in grid.best_params_.items()
    if k.startswith("rf__")
}
print("Best RF regressor params from GridSearchCV:", best_rf_params)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_score.iloc[train_index], y_score.iloc[test_index]

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index]
    Z_test = Z_factors.iloc[test_index]

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = RandomForestRegressor(
        **best_rf_params,
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    mse = mean_squared_error(
        y_test,
        y_pred,
    )
    rmse = np.sqrt(mse)
    rmse_list.append(rmse)

    # Paired Wilcoxon: |y - ŷ_MERF| vs |y - ȳ_train| (per-sample; tests whether MERF fits better than mean baseline)
    baseline = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
    e_merf = np.abs(np.asarray(y_test) - np.asarray(y_pred))
    e_mean = np.abs(np.asarray(y_test) - baseline)
    if len(e_merf) >= 2 and not np.allclose(e_merf, e_mean):
        w_res = wilcoxon(e_merf, e_mean, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    MERF abs errors differ from mean baseline (alpha=0.05).")
    else:
        print("    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse,
        "wilcoxon_p_vs_train_mean_baseline": wilcoxon_p
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_participants": df_clean["participant_id"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "radar_merf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "radar_merf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement sklearn.metrics (from versions: none)
ERROR: No matching distribution found for sklearn.metrics

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Rows: 8515
Participants: 274
Features used: ['Speaking_Rate', 'Articulation_Rate', 'Phonation_Ratio', 'Pause_Rate', 'Pause_Ratio', 'mean_F0', 'stdev_F0_Semitone', 'HNR_dB', 'Spectral_Slope', 'Spectral_Tilt', 'Cepstral_Peak_Prominence', 'mean_F1_Loc', 'std_F1_Loc', 'mean_B1_Loc', 'std_B1_Loc', 'mean_F2_Loc', 'std_F2_Loc', 'mean_B2_Loc', 'std_B2_Loc', 'Spectral_Gravity', 'Spectral_Std_Dev']
Random-effect covariates: ['Age', 'Gender', 'Education_Years', 'Height']
Best RF regressor params from GridSearchCV: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 500}

Processing fold 1


INFO     [merf.py:307] Training GLL is 19879.01739520768 at iteration 1.
INFO     [merf.py:307] Training GLL is 19439.732274562197 at iteration 2.
INFO     [merf.py:307] Training GLL is 19418.167554265092 at iteration 3.
INFO     [merf.py:307] Training GLL is 19284.29472927286 at iteration 4.
INFO     [merf.py:307] Training GLL is 19070.837742843472 at iteration 5.
INFO     [merf.py:307] Training GLL is 18923.48529046682 at iteration 6.
INFO     [merf.py:307] Training GLL is 18761.789006076247 at iteration 7.
INFO     [merf.py:307] Training GLL is 18681.526685638724 at iteration 8.
INFO     [merf.py:307] Training GLL is 18621.257646763905 at iteration 9.
INFO     [merf.py:307] Training GLL is 18552.85117915074 at iteration 10.
INFO     [merf.py:307] Training GLL is 18537.734894551424 at iteration 11.
INFO     [merf.py:307] Training GLL is 18482.47044594131 at iteration 12.
INFO     [merf.py:307] Training GLL is 18485.81679744292 at iteration 13.
INFO     [merf.py:307] Training GLL is 1

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 1: 8.995018631963211e-23
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 6.48

Processing fold 2


INFO     [merf.py:307] Training GLL is 19433.408167510042 at iteration 1.
INFO     [merf.py:307] Training GLL is 18539.523561598628 at iteration 2.
INFO     [merf.py:307] Training GLL is 18343.864927066803 at iteration 3.
INFO     [merf.py:307] Training GLL is 18296.898605611437 at iteration 4.
INFO     [merf.py:307] Training GLL is 18253.701002666057 at iteration 5.
INFO     [merf.py:307] Training GLL is 18222.61040516523 at iteration 6.
INFO     [merf.py:307] Training GLL is 18178.821907948546 at iteration 7.
INFO     [merf.py:307] Training GLL is 18140.15529735988 at iteration 8.
INFO     [merf.py:307] Training GLL is 18143.885022542137 at iteration 9.
INFO     [merf.py:307] Training GLL is 18127.609736201135 at iteration 10.
INFO     [merf.py:307] Training GLL is 18155.31413754007 at iteration 11.
INFO     [merf.py:307] Training GLL is 18147.250320634128 at iteration 12.
INFO     [merf.py:307] Training GLL is 18123.350002768806 at iteration 13.
INFO     [merf.py:307] Training GLL i

...

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 3: 0.005943020430986913
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 6.12

Processing fold 4


INFO     [merf.py:307] Training GLL is 19950.95568775949 at iteration 1.
INFO     [merf.py:307] Training GLL is 19239.045714514457 at iteration 2.
INFO     [merf.py:307] Training GLL is 19131.937965479654 at iteration 3.
INFO     [merf.py:307] Training GLL is 18989.650220130763 at iteration 4.
INFO     [merf.py:307] Training GLL is 18875.3368275731 at iteration 5.
INFO     [merf.py:307] Training GLL is 18782.25738077796 at iteration 6.
INFO     [merf.py:307] Training GLL is 18698.58341689236 at iteration 7.
INFO     [merf.py:307] Training GLL is 18630.37163977468 at iteration 8.
INFO     [merf.py:307] Training GLL is 18580.40558538307 at iteration 9.
INFO     [merf.py:307] Training GLL is 18553.68979016689 at iteration 10.
INFO     [merf.py:307] Training GLL is 18534.991623631337 at iteration 11.
INFO     [merf.py:307] Training GLL is 18524.280793421698 at iteration 12.
INFO     [merf.py:307] Training GLL is 18506.724585989123 at iteration 13.
INFO     [merf.py:307] Training GLL is 185

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 4: 2.4390234743786763e-06
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 5.39

Processing fold 5


INFO     [merf.py:307] Training GLL is 19202.765881904146 at iteration 1.
INFO     [merf.py:307] Training GLL is 18541.48466715344 at iteration 2.
INFO     [merf.py:307] Training GLL is 18536.13098784367 at iteration 3.
INFO     [merf.py:307] Training GLL is 18448.294439764 at iteration 4.
INFO     [merf.py:307] Training GLL is 18282.19051065656 at iteration 5.
INFO     [merf.py:307] Training GLL is 18136.96969263673 at iteration 6.
INFO     [merf.py:307] Training GLL is 18030.58555315243 at iteration 7.
INFO     [merf.py:307] Training GLL is 17946.12686183471 at iteration 8.
INFO     [merf.py:307] Training GLL is 17877.112831126717 at iteration 9.
INFO     [merf.py:307] Training GLL is 17865.888845493293 at iteration 10.
INFO     [merf.py:307] Training GLL is 17821.843987750988 at iteration 11.
INFO     [merf.py:307] Training GLL is 17825.562137751534 at iteration 12.
INFO     [merf.py:307] Training GLL is 17810.321848580592 at iteration 13.
INFO     [merf.py:307] Training GLL is 1780

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 5: 9.813138817314084e-15
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 5.94

Fold results:
   fold  n_train  n_test      rmse  wilcoxon_p_vs_train_mean_baseline
0     1     6812    1703  6.482218                       8.995019e-23
1     2     6812    1703  5.980348                       6.416105e-01
2     3     6812    1703  6.116842                       5.943020e-03
3     4     6812    1703  5.389035                       2.439023e-06
4     5     6812    1703  5.940329                       9.813139e-15

Summary:
  subset  n_rows  n_participants  rmse_mean  rmse_std
0    all    8515             274   5.981754  0.352632

Results saved to:
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\MERF\RADAR


Androids

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from scipy.stats import wilcoxon

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/MERF/ANDROIDS"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["file_stem"] = df["file_stem"].astype(str).str.strip()
df["speech_type"] = df["speech_type"].astype(str).str.strip().str.lower()
df["subgroup_from_path"] = df["subgroup_from_path"].astype(str).str.strip().str.upper()
df["bdi_score"] = pd.to_numeric(df["bdi_score"], errors="coerce")

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
feature_cols = [f"mfcc_{i}" for i in range(1, 14)] + ["pitch_mean", "energy_mean"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = ["speech_type", "subgroup_from_path"]
z_features = [c for c in z_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["bdi_score", "file_stem"] + z_features
df_clean = df.dropna(subset=required_cols).copy()

# -----------------------------
# Build model inputs
# -----------------------------
X = df_clean[feature_cols].astype(float)
y_score = df_clean["bdi_score"].astype(float)
ID_clusters = df_clean["file_stem"]

Z_factors = pd.get_dummies(
    df_clean[z_features],
    drop_first=True
).astype(float)

print("Rows:", len(df_clean))
print("Unique groups:", df_clean["file_stem"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV + simple GridSearchCV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    "rf__n_estimators": [200, 500],
    "rf__max_depth": [None, 10],
    "rf__max_features": ["sqrt"],
    "rf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=gkf,
    n_jobs=-1
)
grid.fit(X, y_score, groups=ID_clusters)

best_rf_params = {
    k.replace("rf__", ""): v
    for k, v in grid.best_params_.items()
    if k.startswith("rf__")
}
print("Best RF regressor params from GridSearchCV:", best_rf_params)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y_score.iloc[train_index].astype(float)
    y_test = y_score.iloc[test_index].astype(float)

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index].astype(float)
    Z_test = Z_factors.iloc[test_index].astype(float)

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = RandomForestRegressor(
        **best_rf_params,
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    rmse_list.append(rmse)

    # Paired Wilcoxon: |y - ŷ_MERF| vs |y - ȳ_train|
    baseline = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
    e_merf = np.abs(np.asarray(y_test) - np.asarray(y_pred))
    e_mean = np.abs(np.asarray(y_test) - baseline)
    if len(e_merf) >= 2 and not np.allclose(e_merf, e_mean):
        w_res = wilcoxon(e_merf, e_mean, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    MERF abs errors differ from mean baseline (alpha=0.05).")
    else:
        print("    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse,
        "wilcoxon_p_vs_train_mean_baseline": wilcoxon_p
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["file_stem"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "androids_merf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "androids_merf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

Rows: 209
Unique groups: 107
Features used: ['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'pitch_mean', 'energy_mean']
Random-effect covariates: ['speech_type', 'subgroup_from_path']
Best RF regressor params from GridSearchCV: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 500}

Processing fold 1


INFO     [merf.py:307] Training GLL is 438.15914673862125 at iteration 1.
INFO     [merf.py:307] Training GLL is 567.9083580770811 at iteration 2.
INFO     [merf.py:307] Training GLL is 590.6263909873893 at iteration 3.
INFO     [merf.py:307] Training GLL is 567.6425781265831 at iteration 4.
INFO     [merf.py:307] Training GLL is 564.8718331754959 at iteration 5.
INFO     [merf.py:307] Training GLL is 579.3024048245563 at iteration 6.
INFO     [merf.py:307] Training GLL is 593.5133640291594 at iteration 7.
INFO     [merf.py:307] Training GLL is 603.4270021063618 at iteration 8.
INFO     [merf.py:307] Training GLL is 611.7950694189514 at iteration 9.
INFO     [merf.py:307] Training GLL is 611.7585001448089 at iteration 10.
INFO     [merf.py:307] Training GLL is 605.600626007351 at iteration 11.
INFO     [merf.py:307] Training GLL is 601.5816692833303 at iteration 12.
INFO     [merf.py:307] Training GLL is 604.7376522351448 at iteration 13.
INFO     [merf.py:307] Training GLL is 600.1369

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 1: 2.302266127188354e-05
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 21.44

Processing fold 2


INFO     [merf.py:307] Training GLL is 371.15655400295844 at iteration 1.
INFO     [merf.py:307] Training GLL is 521.3524090595298 at iteration 2.
INFO     [merf.py:307] Training GLL is 539.6576175370394 at iteration 3.
INFO     [merf.py:307] Training GLL is 545.494892639853 at iteration 4.
INFO     [merf.py:307] Training GLL is 551.1260767372655 at iteration 5.
INFO     [merf.py:307] Training GLL is 562.7216933805406 at iteration 6.
INFO     [merf.py:307] Training GLL is 580.8123930038931 at iteration 7.
INFO     [merf.py:307] Training GLL is 590.2590875314753 at iteration 8.
INFO     [merf.py:307] Training GLL is 598.4499425077955 at iteration 9.
INFO     [merf.py:307] Training GLL is 603.5391268866571 at iteration 10.
INFO     [merf.py:307] Training GLL is 602.269196431832 at iteration 11.
INFO     [merf.py:307] Training GLL is 608.4061783859282 at iteration 12.
INFO     [merf.py:307] Training GLL is 609.0646046606203 at iteration 13.
INFO     [merf.py:307] Training GLL is 607.78622

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 2: 0.9650863012145275
    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).
Fold RMSE = 16.28

Processing fold 3


...

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 3: 0.0013968043307175915
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 10.57

Processing fold 4


INFO     [merf.py:307] Training GLL is 365.3115555315523 at iteration 1.
INFO     [merf.py:307] Training GLL is 521.5529700674006 at iteration 2.
INFO     [merf.py:307] Training GLL is 562.6152875839576 at iteration 3.
INFO     [merf.py:307] Training GLL is 573.2981216978296 at iteration 4.
INFO     [merf.py:307] Training GLL is 581.4491182339966 at iteration 5.
INFO     [merf.py:307] Training GLL is 594.5515375010049 at iteration 6.
INFO     [merf.py:307] Training GLL is 612.6093054811035 at iteration 7.
INFO     [merf.py:307] Training GLL is 632.2250942054077 at iteration 8.
INFO     [merf.py:307] Training GLL is 644.5916643588743 at iteration 9.
INFO     [merf.py:307] Training GLL is 650.3375471973791 at iteration 10.
INFO     [merf.py:307] Training GLL is 664.961571449873 at iteration 11.
INFO     [merf.py:307] Training GLL is 666.7185169265015 at iteration 12.
INFO     [merf.py:307] Training GLL is 667.6212533921051 at iteration 13.
INFO     [merf.py:307] Training GLL is 669.63839

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 4: 0.010927993644709203
    MERF abs errors differ from mean baseline (alpha=0.05).
Fold RMSE = 14.37

Processing fold 5


INFO     [merf.py:307] Training GLL is 423.12097419589105 at iteration 1.
INFO     [merf.py:307] Training GLL is 577.7262128996961 at iteration 2.
INFO     [merf.py:307] Training GLL is 595.206621056634 at iteration 3.
INFO     [merf.py:307] Training GLL is 592.4489666475164 at iteration 4.
INFO     [merf.py:307] Training GLL is 601.7726778219316 at iteration 5.
INFO     [merf.py:307] Training GLL is 614.7398247846702 at iteration 6.
INFO     [merf.py:307] Training GLL is 625.2067278669523 at iteration 7.
INFO     [merf.py:307] Training GLL is 634.0248098992973 at iteration 8.
INFO     [merf.py:307] Training GLL is 643.2393059903023 at iteration 9.
INFO     [merf.py:307] Training GLL is 648.3302465565743 at iteration 10.
INFO     [merf.py:307] Training GLL is 654.276312045361 at iteration 11.
INFO     [merf.py:307] Training GLL is 653.8418600094338 at iteration 12.
INFO     [merf.py:307] Training GLL is 651.7114248165072 at iteration 13.
INFO     [merf.py:307] Training GLL is 645.35233

  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold 5: 0.07059551238762407
    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).
Fold RMSE = 15.16

Fold results:
   fold  n_train  n_test       rmse  wilcoxon_p_vs_train_mean_baseline
0     1      167      42  21.439108                           0.000023
1     2      167      42  16.282437                           0.965086
2     3      167      42  10.566532                           0.001397
3     4      167      42  14.373497                           0.010928
4     5      168      41  15.156822                           0.070596

Summary:
  subset  n_rows  n_groups  rmse_mean  rmse_std
0    all     209       107  15.563679  3.509724

Results saved to:
C:\Users\janku\Documents\KCL\Research Project\Research Project\results\metrics\MERF\ANDROIDS
